# 🎯 LECTURE 11: NAIVE BAYES CLASSIFICATION
## The Probabilistic Powerhouse!

**Goal**: Master Naive Bayes - the algorithm that powers spam filters and text classification

**Topics Covered**:
- **Naive Bayes Theory**: Bayes' theorem and the "naive" assumption
- **Text Classification**: Twitter sentiment analysis (Lecture 11 style)
- **Feature Engineering**: Text preprocessing and vectorization
- **Comparison**: Naive Bayes vs kNN vs Logistic Regression

**Why This Matters**: 
- **HW3 Question 3**: Naive Bayes is the third classification algorithm
- **Text Analysis**: Essential for spam detection, sentiment analysis
- **Probabilistic Thinking**: Foundation for advanced ML concepts


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

print("🎯 LECTURE 11: NAIVE BAYES MASTERY")
print()
print("🧠 BIG PICTURE:")
print("   So far: kNN (similarity-based), Logistic (linear boundary)")
print("   Now: Naive Bayes (probability-based)")
print()
print("🎓 EXAM RELEVANCE:")
print("   • Naive Bayes: HW3 Question 3 (text classification)")
print("   • Bayes' Theorem: Fundamental probability concept")
print("   • Text Processing: Real-world data science skill")
print("   • Algorithm Comparison: Understanding trade-offs")
print()
print("✅ Let's master probabilistic classification!")


In [ ]:
# 🧮 BAYES' THEOREM: The Foundation

## From Probability to Prediction

print("🧮 BAYES' THEOREM: The Mathematical Foundation")
print("The most important formula in probabilistic machine learning!")
print()
print("📋 THE FORMULA:")
print("   P(A|B) = P(B|A) × P(A) / P(B)")
print()
print("🎯 FOR CLASSIFICATION:")
print("   P(Class|Features) = P(Features|Class) × P(Class) / P(Features)")
print()
print("🧠 IN PLAIN ENGLISH:")
print("   'What's the probability of this class, given these features?'")
print()
print("🌟 NAIVE ASSUMPTION:")
print("   Features are independent (that's why it's 'naive')")
print("   P(word1, word2|spam) = P(word1|spam) × P(word2|spam)")

# Simple example to illustrate Bayes' theorem
print(f"\n📊 SIMPLE EXAMPLE: Email Classification")
print(f"Let's classify an email as SPAM or HAM (not spam)")
print()

# Create a simple dataset
emails = [
    ("Buy now! Limited offer!", "spam"),
    ("Meeting at 3pm today", "ham"),
    ("Free money! Click here!", "spam"),
    ("How was your weekend?", "ham"),
    ("Urgent! Act now!", "spam"),
    ("Lunch tomorrow?", "ham"),
    ("Win big! Casino!", "spam"),
    ("Project deadline reminder", "ham")
]

# Extract words and labels
all_words = []
labels = []
for email, label in emails:
    words = email.lower().split()
    all_words.extend(words)
    labels.append(label)

# Count occurrences
spam_count = labels.count("spam")
ham_count = labels.count("ham")
total_emails = len(emails)

print(f"Dataset:")
print(f"   Total emails: {total_emails}")
print(f"   Spam emails: {spam_count}")
print(f"   Ham emails: {ham_count}")
print(f"   P(spam) = {spam_count/total_emails:.2f}")
print(f"   P(ham) = {ham_count/total_emails:.2f}")

# Let's classify a new email: "Free offer!"
test_email = "Free offer!"
test_words = test_email.lower().split()

print(f"\n🧪 CLASSIFY NEW EMAIL: '{test_email}'")
print(f"Words: {test_words}")

# Calculate P(word|spam) and P(word|ham) for each word
spam_emails = [email for email, label in emails if label == "spam"]
ham_emails = [email for email, label in emails if label == "ham"]

print(f"\n📊 WORD PROBABILITIES:")
for word in test_words:
    # Count word in spam emails
    spam_word_count = sum(1 for email in spam_emails if word in email.lower())
    ham_word_count = sum(1 for email in ham_emails if word in email.lower())
    
    # Add smoothing (Laplace smoothing)
    p_word_spam = (spam_word_count + 1) / (spam_count + 2)
    p_word_ham = (ham_word_count + 1) / (ham_count + 2)
    
    print(f"   '{word}': P(word|spam)={p_word_spam:.3f}, P(word|ham)={p_word_ham:.3f}")

print(f"\n🎯 NAIVE BAYES PREDICTION:")
print(f"   P(spam|'free offer') ∝ P('free'|spam) × P('offer'|spam) × P(spam)")
print(f"   P(ham|'free offer') ∝ P('free'|ham) × P('offer'|ham) × P(ham)")
print(f"   Compare these probabilities → higher one wins!")
print(f"\n💡 This is exactly what sklearn's MultinomialNB does!")


In [ ]:
## 🐦 Text Classification: Twitter Data Science Detection

print("🐦 TWITTER CLASSIFICATION: Lecture 11 Style!")
print("Detecting data science tweets using Naive Bayes")

# Create synthetic Twitter-like data (similar to Lecture 11)
# This mimics the QuantumTunnel dataset structure
tweets_data = [
    # Data Science Related Tweets
    ("Machine learning is revolutionizing healthcare #AI #DataScience", 1),
    ("Just finished my Python data analysis project! #coding #analytics", 1),
    ("Statistical modeling helps predict customer behavior #statistics #ML", 1),
    ("Working with pandas dataframes today #python #dataanalysis", 1),
    ("Neural networks are fascinating! Deep learning workshop tomorrow", 1),
    ("Big data analytics reveals interesting patterns #bigdata #insights", 1),
    ("Regression analysis shows strong correlation #statistics #research", 1),
    ("Data visualization with matplotlib and seaborn #dataviz #python", 1),
    ("Cross-validation improves model performance #machinelearning #ML", 1),
    ("Feature engineering is crucial for good models #datascience #AI", 1),
    
    # Non-Data Science Tweets
    ("Beautiful sunset today! Perfect weather for a walk #nature #photography", 0),
    ("Just watched an amazing movie! Highly recommend it #entertainment #film", 0),
    ("Cooking dinner with friends tonight #food #friendship #cooking", 0),
    ("Traffic is terrible this morning #commute #city #transportation", 0),
    ("Weekend plans: hiking and relaxing #weekend #outdoors #hiking", 0),
    ("New restaurant opened downtown! Great food #restaurant #food #local", 0),
    ("Concert was incredible last night! #music #concert #entertainment", 0),
    ("Planning vacation to Europe next summer #travel #vacation #europe", 0),
    ("Morning coffee and newspaper routine #coffee #morning #news", 0),
    ("Sports game tonight! Go team! #sports #game #excitement", 0)
]

# Convert to DataFrame
df_tweets = pd.DataFrame(tweets_data, columns=['Tweet', 'DataScience'])

print(f"\n📊 Twitter Dataset:")
print(f"   Total tweets: {len(df_tweets)}")
print(f"   Data Science tweets: {df_tweets['DataScience'].sum()}")
print(f"   Non-Data Science tweets: {(df_tweets['DataScience'] == 0).sum()}")
print(f"\n📝 Sample tweets:")
print(df_tweets.head(3))

# Text preprocessing function (from Lecture 11)
def preprocess_tweet(tweet):
    """Clean tweet text by removing URLs and hashtags"""
    # Remove URLs
    tweet = re.sub(r"http\S+", "", tweet)
    # Remove hashtag symbols (keep the words)
    tweet = re.sub(r"#", "", tweet)
    # Convert to lowercase
    tweet = tweet.lower()
    # Remove extra whitespace
    tweet = re.sub(r'\s+', ' ', tweet).strip()
    return tweet

# Apply preprocessing
df_tweets['Tweet_Clean'] = df_tweets['Tweet'].apply(preprocess_tweet)

print(f"\n🧹 PREPROCESSING EXAMPLE:")
print(f"Original: {df_tweets['Tweet'].iloc[0]}")
print(f"Cleaned:  {df_tweets['Tweet_Clean'].iloc[0]}")

# Split the data
X_text = df_tweets['Tweet_Clean']
y_text = df_tweets['DataScience']

X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X_text, y_text, test_size=0.3, random_state=42)

print(f"\n📋 Data Split:")
print(f"   Training tweets: {len(X_train_text)}")
print(f"   Test tweets: {len(X_test_text)}")

# Convert text to numerical features using CountVectorizer
vectorizer = CountVectorizer(stop_words='english', max_features=100)
X_train_vec = vectorizer.fit_transform(X_train_text)
X_test_vec = vectorizer.transform(X_test_text)

print(f"\n🔢 VECTORIZATION:")
print(f"   Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"   Feature matrix shape: {X_train_vec.shape}")
print(f"   Sample feature names: {list(vectorizer.get_feature_names_out())[:10]}")

# Train Naive Bayes classifier
nb_classifier = MultinomialNB()
nb_classifier.fit(X_train_vec, y_train_text)

# Make predictions
y_pred_text = nb_classifier.predict(X_test_vec)
y_pred_proba_text = nb_classifier.predict_proba(X_test_vec)

# Calculate accuracy
accuracy_text = accuracy_score(y_test_text, y_pred_text)

print(f"\n🎯 NAIVE BAYES RESULTS:")
print(f"   Training accuracy: {nb_classifier.score(X_train_vec, y_train_text):.3f}")
print(f"   Test accuracy: {accuracy_text:.3f}")

# Show some predictions
print(f"\n🔍 EXAMPLE PREDICTIONS:")
for i in range(min(5, len(X_test_text))):
    tweet = X_test_text.iloc[i]
    actual = y_test_text.iloc[i]
    predicted = y_pred_text[i]
    probability = y_pred_proba_text[i][1]  # Probability of being data science
    
    print(f"\nTweet: '{tweet[:60]}{'...' if len(tweet) > 60 else ''}'")
    print(f"   Actual: {'Data Science' if actual == 1 else 'Not Data Science'}")
    print(f"   Predicted: {'Data Science' if predicted == 1 else 'Not Data Science'}")
    print(f"   Confidence: {probability:.3f}")
    print(f"   {'✅ Correct' if predicted == actual else '❌ Wrong'}")


In [ ]:
## 🔍 Feature Analysis: What Makes a Data Science Tweet?

print("🔍 FEATURE ANALYSIS: Understanding the Model")
print("What words indicate a data science tweet?")

# Get feature names and their importance
feature_names = vectorizer.get_feature_names_out()
log_probs = nb_classifier.feature_log_prob_

# Calculate the difference in log probabilities
# Higher difference means more discriminative for data science
prob_diff = log_probs[1] - log_probs[0]  # DataScience - NotDataScience

# Get top features for each class
top_datascience_indices = np.argsort(prob_diff)[-10:][::-1]
top_notdatascience_indices = np.argsort(prob_diff)[:10]

print(f"\n📊 TOP WORDS FOR DATA SCIENCE TWEETS:")
for i, idx in enumerate(top_datascience_indices):
    word = feature_names[idx]
    score = prob_diff[idx]
    print(f"   {i+1}. '{word}' (score: {score:.3f})")

print(f"\n📊 TOP WORDS FOR NON-DATA SCIENCE TWEETS:")
for i, idx in enumerate(top_notdatascience_indices):
    word = feature_names[idx]
    score = prob_diff[idx]
    print(f"   {i+1}. '{word}' (score: {score:.3f})")

# Visualize the results
plt.figure(figsize=(15, 10))

# Plot 1: Confusion Matrix
plt.subplot(2, 3, 1)
cm_text = confusion_matrix(y_test_text, y_pred_text)
sns.heatmap(cm_text, annot=True, fmt='d', cmap='Blues',
           xticklabels=['Not Data Science', 'Data Science'], 
           yticklabels=['Not Data Science', 'Data Science'])
plt.title('🎯 Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

# Plot 2: Feature importance (top data science words)
plt.subplot(2, 3, 2)
top_ds_words = [feature_names[idx] for idx in top_datascience_indices[:5]]
top_ds_scores = [prob_diff[idx] for idx in top_datascience_indices[:5]]
plt.barh(top_ds_words, top_ds_scores, color='skyblue')
plt.title('🔬 Top Data Science Words')
plt.xlabel('Discriminative Score')

# Plot 3: Feature importance (top non-data science words)
plt.subplot(2, 3, 3)
top_nds_words = [feature_names[idx] for idx in top_notdatascience_indices[:5]]
top_nds_scores = [prob_diff[idx] for idx in top_notdatascience_indices[:5]]
plt.barh(top_nds_words, top_nds_scores, color='lightcoral')
plt.title('🌟 Top Non-Data Science Words')
plt.xlabel('Discriminative Score')

# Plot 4: Prediction probabilities distribution
plt.subplot(2, 3, 4)
ds_probs = y_pred_proba_text[y_test_text == 1, 1]  # Data science tweets
nds_probs = y_pred_proba_text[y_test_text == 0, 1]  # Non-data science tweets

plt.hist(ds_probs, alpha=0.7, label='Actually Data Science', bins=10, color='blue')
plt.hist(nds_probs, alpha=0.7, label='Actually Not Data Science', bins=10, color='red')
plt.axvline(x=0.5, color='black', linestyle='--', label='Decision Threshold')
plt.xlabel('Predicted Probability (Data Science)')
plt.ylabel('Count')
plt.title('📈 Prediction Probabilities')
plt.legend()

# Plot 5: Class distribution
plt.subplot(2, 3, 5)
class_counts = [df_tweets['DataScience'].value_counts()[0], df_tweets['DataScience'].value_counts()[1]]
plt.pie(class_counts, labels=['Not Data Science', 'Data Science'], autopct='%1.1f%%', 
        colors=['lightcoral', 'skyblue'])
plt.title('📊 Class Distribution')

# Plot 6: Vocabulary size effect
plt.subplot(2, 3, 6)
vocab_sizes = [10, 25, 50, 100, 200]
accuracies = []

for vocab_size in vocab_sizes:
    vec_temp = CountVectorizer(stop_words='english', max_features=vocab_size)
    X_temp = vec_temp.fit_transform(X_train_text)
    X_test_temp = vec_temp.transform(X_test_text)
    
    nb_temp = MultinomialNB()
    nb_temp.fit(X_temp, y_train_text)
    acc = nb_temp.score(X_test_temp, y_test_text)
    accuracies.append(acc)

plt.plot(vocab_sizes, accuracies, 'bo-', linewidth=2, markersize=6)
plt.xlabel('Vocabulary Size')
plt.ylabel('Test Accuracy')
plt.title('📈 Vocabulary Size vs Accuracy')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✅ INSIGHTS:")
print(f"   • Words like 'learning', 'data', 'python' strongly indicate data science")
print(f"   • Words like 'food', 'movie', 'weekend' indicate non-data science")
print(f"   • Naive Bayes naturally finds discriminative features")
print(f"   • Vocabulary size affects performance - need to tune!")


In [ ]:
## 🆚 Algorithm Showdown: kNN vs Logistic vs Naive Bayes

print("🆚 THE ULTIMATE CLASSIFICATION COMPARISON")
print("All three algorithms on the same dataset!")

# For fair comparison, let's create numerical features from the text
# We'll use the same vectorized features for all algorithms

print(f"\n🏁 ALGORITHM COMPARISON:")
print(f"Dataset: Twitter data science classification")
print(f"Features: {X_train_vec.shape[1]} word counts")
print(f"Training samples: {X_train_vec.shape[0]}")
print(f"Test samples: {X_test_vec.shape[0]}")

# Train all three algorithms
algorithms = {
    'kNN (k=3)': KNeighborsClassifier(n_neighbors=3),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Naive Bayes': MultinomialNB()
}

results = {}

print(f"\n🧪 TRAINING AND EVALUATION:")
for name, model in algorithms.items():
    # Train the model
    model.fit(X_train_vec, y_train_text)
    
    # Make predictions
    y_pred = model.predict(X_test_vec)
    y_pred_proba = model.predict_proba(X_test_vec)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Calculate metrics
    train_acc = model.score(X_train_vec, y_train_text)
    test_acc = accuracy_score(y_test_text, y_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_vec, y_train_text, cv=3)
    
    results[name] = {
        'train_acc': train_acc,
        'test_acc': test_acc,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"\n{name}:")
    print(f"   Train accuracy: {train_acc:.3f}")
    print(f"   Test accuracy: {test_acc:.3f}")
    print(f"   CV accuracy: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")

# Find the best algorithm
best_algo = max(results.keys(), key=lambda x: results[x]['test_acc'])
print(f"\n🏆 WINNER: {best_algo} with {results[best_algo]['test_acc']:.3f} accuracy!")

# Detailed comparison visualization
plt.figure(figsize=(15, 10))

# Plot 1: Accuracy comparison
plt.subplot(2, 3, 1)
algo_names = list(results.keys())
test_accs = [results[name]['test_acc'] for name in algo_names]
cv_means = [results[name]['cv_mean'] for name in algo_names]

x_pos = np.arange(len(algo_names))
plt.bar(x_pos - 0.2, test_accs, 0.4, label='Test Accuracy', alpha=0.8, color='skyblue')
plt.bar(x_pos + 0.2, cv_means, 0.4, label='CV Accuracy', alpha=0.8, color='lightgreen')

plt.xlabel('Algorithm')
plt.ylabel('Accuracy')
plt.title('🏆 Accuracy Comparison')
plt.xticks(x_pos, [name.split('(')[0].strip() for name in algo_names], rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2-4: Confusion matrices for each algorithm
for i, (name, result) in enumerate(results.items()):
    plt.subplot(2, 3, i+2)
    cm = confusion_matrix(y_test_text, result['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=['Not DS', 'DS'], yticklabels=['Not DS', 'DS'])
    plt.title(f'{name.split("(")[0].strip()}\nAcc: {result["test_acc"]:.3f}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')

# Plot 5: Algorithm characteristics table
plt.subplot(2, 3, 5)
plt.axis('off')

comparison_data = [
    ['Characteristic', 'kNN', 'Logistic Reg', 'Naive Bayes'],
    ['Training Speed', 'Fast', 'Medium', 'Fast'],
    ['Prediction Speed', 'Slow', 'Fast', 'Fast'],
    ['Memory Usage', 'High', 'Low', 'Low'],
    ['Interpretability', 'Low', 'High', 'Medium'],
    ['Assumptions', 'None', 'Linear boundary', 'Independence'],
    ['Text Data', 'Poor', 'Good', 'Excellent'],
    ['Overfitting Risk', 'Medium', 'Low', 'Low'],
    ['Hyperparameters', 'k', 'C, penalty', 'smoothing']
]

table = plt.table(cellText=comparison_data[1:], colLabels=comparison_data[0],
                 cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.5)

# Color header
for i in range(4):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white')

plt.title('📊 Algorithm Comparison', pad=20, fontsize=12, fontweight='bold')

# Plot 6: Performance vs training size
plt.subplot(2, 3, 6)
train_sizes = [5, 8, 10, 12, 14]  # Different training sizes
knn_scores = []
logistic_scores = []
nb_scores = []

for size in train_sizes:
    if size <= len(X_train_text):
        # Use subset of training data
        X_subset = X_train_vec[:size]
        y_subset = y_train_text.iloc[:size]
        
        # Train models
        knn_temp = KNeighborsClassifier(n_neighbors=3)
        logistic_temp = LogisticRegression(random_state=42, max_iter=1000)
        nb_temp = MultinomialNB()
        
        knn_temp.fit(X_subset, y_subset)
        logistic_temp.fit(X_subset, y_subset)
        nb_temp.fit(X_subset, y_subset)
        
        # Evaluate
        knn_scores.append(knn_temp.score(X_test_vec, y_test_text))
        logistic_scores.append(logistic_temp.score(X_test_vec, y_test_text))
        nb_scores.append(nb_temp.score(X_test_vec, y_test_text))

plt.plot(train_sizes, knn_scores, 'o-', label='kNN', linewidth=2)
plt.plot(train_sizes, logistic_scores, 's-', label='Logistic', linewidth=2)
plt.plot(train_sizes, nb_scores, '^-', label='Naive Bayes', linewidth=2)
plt.xlabel('Training Size')
plt.ylabel('Test Accuracy')
plt.title('📈 Learning Curves')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n🎯 KEY INSIGHTS:")
print(f"   • Naive Bayes excels at text classification")
print(f"   • kNN struggles with high-dimensional sparse text features")
print(f"   • Logistic Regression provides good balance and interpretability")
print(f"   • For text data: Naive Bayes > Logistic Regression > kNN")
print(f"   • Choice depends on dataset characteristics and requirements")


In [ ]:
## 🎓 EXAM MASTERY CHECKLIST

print("🎓 LECTURE 11: EXAM MASTERY CHECKLIST")
print()
print("✅ NAIVE BAYES MASTERY:")
print("   □ Can you explain Bayes' theorem in simple terms?")
print("   □ Do you understand the 'naive' independence assumption?")
print("   □ Can you explain why Naive Bayes works well for text?")
print("   □ Do you know how to preprocess text data?")
print("   □ Can you implement text classification with sklearn?")
print()
print("✅ TEXT PROCESSING MASTERY:")
print("   □ Can you use CountVectorizer to convert text to numbers?")
print("   □ Do you understand term-document matrices?")
print("   □ Can you preprocess text (remove URLs, hashtags)?")
print("   □ Do you know about stop words and vocabulary size?")
print("   □ Can you interpret feature importance in text models?")
print()
print("✅ ALGORITHM COMPARISON MASTERY:")
print("   □ Can you compare kNN, Logistic, and Naive Bayes?")
print("   □ Do you know when to use each algorithm?")
print("   □ Can you evaluate models with cross-validation?")
print("   □ Do you understand the trade-offs between algorithms?")
print("   □ Can you choose the best algorithm for a given problem?")
print()
print("🎯 KEY FORMULAS TO REMEMBER:")
print("   • Bayes' Theorem: P(A|B) = P(B|A) × P(A) / P(B)")
print("   • Naive assumption: P(x1,x2|y) = P(x1|y) × P(x2|y)")
print("   • Classification: argmax P(y|x) = argmax P(x|y) × P(y)")
print("   • Laplace smoothing: (count + 1) / (total + vocabulary_size)")
print()
print("🧠 CONCEPTUAL UNDERSTANDING:")
print("   • Probabilistic vs geometric vs similarity-based classification")
print("   • Independence assumption makes computation tractable")
print("   • Text as high-dimensional sparse feature vectors")
print("   • Feature engineering importance for text data")

print("\n" + "="*60)
print("🏆 FINAL CHALLENGE: HW3 Complete Preview!")
print()
print("📋 SCENARIO (HW3 Style):")
print("   You have three datasets: numerical, text, and mixed")
print("   Goal: Apply all three classification algorithms")
print("   Compare performance and choose the best approach")
print()
print("❓ QUESTIONS:")
print("   1. Which algorithm for high-dimensional text data?")
print("   2. How do you preprocess text for classification?")
print("   3. What's the naive assumption in Naive Bayes?")
print("   4. How do you compare algorithms fairly?")
print("   5. When would Naive Bayes outperform kNN and Logistic?")
print()
print("💡 ANSWERS:")
print("   1. Naive Bayes (designed for text, handles sparsity well)")
print("   2. Remove URLs/hashtags, lowercase, vectorize, remove stop words")
print("   3. Features are conditionally independent given the class")
print("   4. Same train/test split, cross-validation, multiple metrics")
print("   5. Text data, categorical features, small datasets, fast predictions needed")
print()
print("✅ If you got these right, you've mastered all HW3 algorithms!")

# Create a comprehensive practice problem
print(f"\n🧠 COMPREHENSIVE PRACTICE PROBLEM:")
print(f"Let's solve a complete classification pipeline!")

# Generate a mixed dataset (numerical + categorical)
np.random.seed(42)

# Create synthetic dataset with both numerical and categorical features
n_samples = 200
age = np.random.normal(35, 10, n_samples)
income = np.random.normal(50000, 15000, n_samples)
education = np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples)
city = np.random.choice(['NYC', 'LA', 'Chicago', 'Houston'], n_samples)

# Create target based on features (will buy premium product)
buy_premium = (
    (age > 30) & (income > 45000) & 
    (np.isin(education, ['Master', 'PhD'])) & 
    (np.isin(city, ['NYC', 'LA']))
).astype(int)

# Add some noise
noise = np.random.binomial(1, 0.1, n_samples)  # 10% noise
buy_premium = np.logical_xor(buy_premium, noise).astype(int)

# Create DataFrame
df_practice = pd.DataFrame({
    'age': age,
    'income': income,
    'education': education,
    'city': city,
    'buy_premium': buy_premium
})

print(f"\nDataset: Customer premium product purchase prediction")
print(f"   Samples: {len(df_practice)}")
print(f"   Features: age (numerical), income (numerical), education (categorical), city (categorical)")
print(f"   Target: buy_premium (0/1)")
print(f"   Premium buyers: {buy_premium.sum()} ({buy_premium.mean()*100:.1f}%)")

# Prepare features for different algorithms
from sklearn.preprocessing import LabelEncoder, StandardScaler

# For Naive Bayes and Logistic: encode categorical variables
le_education = LabelEncoder()
le_city = LabelEncoder()

X_encoded = pd.DataFrame({
    'age': df_practice['age'],
    'income': df_practice['income'],
    'education': le_education.fit_transform(df_practice['education']),
    'city': le_city.fit_transform(df_practice['city'])
})

# For kNN: also scale numerical features
scaler = StandardScaler()
X_scaled = X_encoded.copy()
X_scaled[['age', 'income']] = scaler.fit_transform(X_encoded[['age', 'income']])

y_practice = df_practice['buy_premium']

# Split data
X_train_prac, X_test_prac, y_train_prac, y_test_prac = train_test_split(
    X_encoded, y_practice, test_size=0.3, random_state=42)

X_train_scaled, X_test_scaled, _, _ = train_test_split(
    X_scaled, y_practice, test_size=0.3, random_state=42)

print(f"\nStep 1: Data preprocessing complete")
print(f"   Encoded categorical variables")
print(f"   Scaled features for kNN")
print(f"   Train/test split: {len(X_train_prac)}/{len(X_test_prac)}")

# Apply all three algorithms
models_practice = {
    'kNN (k=5)': (KNeighborsClassifier(n_neighbors=5), X_train_scaled, X_test_scaled),
    'Logistic Regression': (LogisticRegression(random_state=42), X_train_prac, X_test_prac),
    'Naive Bayes': (GaussianNB(), X_train_prac, X_test_prac)  # Gaussian for numerical features
}

print(f"\nStep 2: Algorithm comparison")
best_score = 0
best_model_name = ""

for name, (model, X_train, X_test) in models_practice.items():
    # Train and evaluate
    model.fit(X_train, y_train_prac)
    train_score = model.score(X_train, y_train_prac)
    test_score = model.score(X_test, y_test_prac)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train_prac, cv=5)
    
    print(f"\n{name}:")
    print(f"   Train accuracy: {train_score:.3f}")
    print(f"   Test accuracy: {test_score:.3f}")
    print(f"   CV accuracy: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")
    
    if test_score > best_score:
        best_score = test_score
        best_model_name = name

print(f"\nStep 3: Results")
print(f"🏆 Best algorithm: {best_model_name} ({best_score:.3f} accuracy)")
print(f"✅ You just completed a full ML pipeline with all three algorithms!")
print(f"🎯 This is exactly what HW3 asks you to do!")
